# Unit Test Suite: Environment & Component Verification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ltellado/LAQuer-Audit/blob/main/notebooks/tests/laquer-audit-unit-tests.ipynb)

This notebook executes isolated unit tests for the core modules supporting the project **"Reproducing and Extending LAQuer for Post-Hoc Attribution"**.

### Objectives
1. **Environment Sanity**: Validate platform detection, directory structure, baseline submodule integrity (`eranhirs/LAQuer` pinned at commit `a4affb2`), credential retrieval, and PyTorch hardware acceleration (CUDA/MPS/CPU).
2. **Component Isolation**: Exercise each pipeline stage individually to verify API contracts, string normalization, and verification invariants before running full-scale experimental batches.

---

In [ ]:
# -------------------------------------------------------------------------
# Environment Setup: Anchor working directory & sync latest code
# -------------------------------------------------------------------------
import os, sys

if "google.colab" in sys.modules:
    repo_dir = "/content/LAQuer-Audit"
    if not os.path.exists(repo_dir):
        !git clone https://github.com/ltellado/LAQuer-Audit.git {repo_dir}
        %cd {repo_dir}
        !bash setup.sh
    else:
        %cd {repo_dir}
        # Discard any local modifications and pull latest remote commits
        !git reset --hard HEAD
        !git pull origin main
else:
    # Local execution (VS Code): traverse up until repo root (with setup.sh)
    cur = os.getcwd()
    while cur != "/" and not os.path.exists(os.path.join(cur, "setup.sh")):
        cur = os.path.dirname(cur)
    if os.path.exists(os.path.join(cur, "setup.sh")):
        %cd {cur}

print("Active Working Directory:", os.getcwd())

## Test 1: `LAQuerAudit` Environment Bootstrap (`src/bootstrap.py`)

Validates the static bootstrapping utility:
- **Platform Detection**: Distinguishes between local runtimes (VS Code / CLI) and Google Colab.
- **Repository Root Resolution**: Dynamically discovers root paths and registers `src/` on `sys.path`.
- **Baseline Submodule Integrity**: Confirms the pinned `LAQuer` submodule exists.
- **Credential Access**: Loads `OPENAI_API_KEY` from the environment or Colab Secrets (`userdata`).
- **Device Arbitration**: Resolves the optimal compute device (`cuda`, `mps`, or `cpu`).

**Pass Criteria**: Outputs `✓ Unit test passed for LAQuerAudit!` with no unhandled exceptions.

In [ ]:
# Test LAQuerAudit
%run src/bootstrap.py

## Test 2: `LAQuerLocalizer` LLM In-Context Localizer (`src/localizer.py`)

Tests the core LLM span-extraction module (Hirsch et al., ACL 2025):
- **Prompt Construction**: Compiles the official instruction prompt (`ins_prompt`) and the 3 LFQA few-shot exemplars with target source documents and claim.
- **Deterministic Inference**: Dispatches to OpenAI chat completion with `temperature=0`.
- **Boundary Stripping**: Normalizes trailing boundary punctuation (`clean_span_boundary`) that commercial chat models append to mid-sentence spans.
- **Extractive Invariant Verification**: Asserts that every extracted span matches at least one source document verbatim as an exact substring.

**Pass Criteria**: Outputs parsed spans, reports `All Verbatim Match: True`, and concludes with `✓ Unit test passed for LAQuerLocalizer!`.

In [ ]:
# Test LAQuerLocalizer
%run src/localizer.py

## Test 3: `LAQuerVerifier` NLI Cross-Encoder (`src/verifier.py`)

Tests the cross-encoder verification module using DeBERTa-v3 (`cross-encoder/nli-deberta-v3-base`):
- **Hardware Acceleration**: Confirms automatic device routing via `LAQuerAudit.resolve_device()` (`cuda` in Colab, `mps` on Apple Silicon, or `cpu`).
- **NLI Classification**: Evaluates pairwise classification probabilities across three distinct relationships:
  1. *Entailment* (supporting attribution span entails claim)
  2. *Contradiction* (conflicting factual claims)
  3. *Neutral* (unrelated or unsupported premises)
- **Batch Processing**: Exercises `batch_verify` for accelerated multi-premise inference.

**Pass Criteria**: Correctly predicts class labels for all three test cases and prints `✓ Unit test passed for LAQuerVerifier!`.

In [ ]:
# Test LAQuerVerifier
%run src/verifier.py

## Test 4: `LAQuerDecomposer` FActScore Atomic Decomposition (`src/decomposer.py`)

Tests the highlight synthesis module following Min et al. (EMNLP 2023) and Choi et al. (TACL 2021):
- **Atomic Fact Decomposition**: Breaks down complex compound sentences into single-assertion propositions.
- **Molecular Decontextualization**: Resolves ambiguous pronouns (`they`, `he`, `it`) against the paragraph context so claims stand alone for NLI verification.

**Pass Criteria**: Outputs $\ge 2$ atomic facts, resolves antecedent pronouns, and prints `✓ Unit test passed for LAQuerDecomposer!`.

In [ ]:
# Test LAQuerDecomposer
%run src/decomposer.py

## Test 5: `LAQuerGenerator` Unified Text Generation (`src/generator.py`)

Tests the polymorphic generation module using `gpt-4o-mini` with deterministic inference (`temperature=0`):
- **Grounded Mode**: Generates citation-suppressed responses conditioned on reference documents.
- **Parametric Mode (None & Empty List)**: Generates ungrounded responses directly from parametric weights when sources are omitted.
- **Logging**: Displays query strings and token metrics across all three conditions.

**Pass Criteria**: Correctly flags `is_grounded` state across all three tests and prints `✓ Unit test passed for unified LAQuerGenerator!`.

In [ ]:
# Test LAQuerGenerator
%run src/generator.py

## Test 6: `LAQuerDataset` Benchmark Data Governance (`src/dataset.py`)

Verifies the unified dataset preparation and disk-caching layer:
- **LFQA Benchmark (Reproduction)**: Extracts the 45 evaluation document sets from upstream `LAQuer/data/LFQA/test.json`, matches each `query_id` against the streaming annotations from Liu et al. (2023), and materializes `data/LFQA/test.json`.
- **ASQA Benchmark (Extension)**: Slices 50 ambiguous questions with reference gold Wikipedia passages from the Hugging Face `din0s/asqa` split (or ALCE fallback) into `data/ASQA/test.json`.
- **Validation**: Enforces existence and non-empty status for both target evaluation artifacts.

**Pass Criteria**: Outputs `LFQA Instances: 45`, `ASQA Instances: 50`, and prints `✓ Unit test passed for LAQuerDataset!`.

In [ ]:
# Test LAQuerDataset
%run src/dataset.py

## Test 7: `LAQuerRetriever` Multi-Corpus Knowledge Retrieval (`src/retriever.py`)

Validates evidence passage acquisition across all three supported corpora:
- **`Wikipedia` (Default)**: Live dynamic search via the Wikimedia REST API with paragraph segmentation (`prepend_title=False`).
- **`LFQA`**: Static benchmark lookup over the 45 evaluation sets in `data/LFQA/test.json`.
- **`ASQA`**: Benchmark lookup over the 50 gold reference sets in `data/ASQA/test.json`.
- **Corpus Validation**: Verifies `CORPORA = ['WIKIPEDIA', 'LFQA', 'ASQA']` and ensures invalid corpus requests raise `ValueError`.

**Pass Criteria**: Outputs retrieved passage counts for all three corpora and prints `✓ Unit test passed for LAQuerRetriever!`.

In [ ]:
# Test LAQuerRetriever
%run src/retriever.py